# 08 — Multidimensional Disagreement Patterns

The earlier analyses establish several different forms of evidence: exact and normalized string comparison, embedding geometry, prompt sensitivity, service behavior, script signals, and the claims models make in their rationales. I bring those layers together here without forcing every language into one causal category.

The unit remains the language-code row. `comparison_status` describes the **degree** of string variation. The additional fields describe its **shape**, **participants**, and **context**. These pattern labels are descriptive and non-exclusive; they identify cases for interpretation rather than deciding whether a translation is correct.

## 8.1 Build and load the pattern table

`build_disagreement_patterns.py` combines the neutral comparison table with prompt-level translations and rationale signals. It writes `disagreement_patterns.csv`, which can be regenerated independently of this notebook.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

alt.data_transformers.enable('vegafusion')
sys.path.insert(0, str(Path('..').resolve()))

from scripts.utils import get_data_directory_path, read_csv_file, save_chart

DATA_DIR = get_data_directory_path()
TERM_SLUG = 'digital_humanities'
EVAL_DIR = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
PATTERN_PATH = os.path.join(EVAL_DIR, 'disagreement_patterns.csv')
FIGURE_DIR = Path('..') / 'figures' / 'dhq_article'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

proc = subprocess.run(
    [sys.executable, 'scripts/exploration/build_disagreement_patterns.py'],
    cwd=str(Path('..').resolve()), capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode:
    print(proc.stderr)
    raise RuntimeError('build_disagreement_patterns.py failed')

patterns = read_csv_file(PATTERN_PATH)
assert len(patterns) == 881
print(f'{len(patterns):,} language-code rows; {len(patterns.columns)} pattern and evidence columns')

Retrieving translation pipeline data directory path...

Retrieving translation pipeline data directory path...
Saved 881 rows x 61 columns -> /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_patterns.csv
Saved 20661 rationale rows -> /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/rationale_signal_detail.csv
disagreement_shape
fragmented_disagreement            746
dominant_form_with_alternatives    128
unanimous                            5
near_unanimous_with_outlier          2

881 language-code rows; 61 pattern and evidence columns


## 8.2 Degree and shape are different questions

The three comparison statuses measure how far strings differ. `disagreement_shape` instead uses the sizes of exact normalized-form clusters:

- **Unanimous**: one form across all usable services.
- **Near-unanimous with outlier**: all but one service share a form.
- **Dominant form with alternatives**: at least half of services share one form.
- **Two-cluster split**: two comparably sized form clusters.
- **Fragmented disagreement**: no form reaches half of the available services.

This distinction separates broad fragmentation from cases where apparent variation is driven by one or two services.

In [2]:
SHAPE_ORDER = [
    'unanimous', 'near_unanimous_with_outlier',
    'dominant_form_with_alternatives', 'two_cluster_split',
    'fragmented_disagreement', 'insufficient_comparison',
]
SHAPE_LABELS = {s: s.replace('_', ' ').title() for s in SHAPE_ORDER}

shape_counts = patterns['disagreement_shape'].value_counts().rename_axis('disagreement_shape').reset_index(name='languages')
shape_counts['label'] = shape_counts['disagreement_shape'].map(SHAPE_LABELS)
shape_counts['pct'] = shape_counts['languages'] / len(patterns)
shape_counts['annotation'] = shape_counts.apply(lambda r: f"{r['languages']:,} ({r['pct']:.1%})", axis=1)
display(shape_counts)

shape_bar = alt.Chart(shape_counts).mark_bar().encode(
    y=alt.Y('label:N', sort=[SHAPE_LABELS[s] for s in SHAPE_ORDER], title=None),
    x=alt.X('languages:Q', title='Language-code rows'),
    color=alt.Color('disagreement_shape:N', legend=None, scale=alt.Scale(scheme='tableau10')),
)
shape_text = shape_bar.mark_text(align='left', dx=5, color='black').encode(text='annotation:N')
shape_chart = (shape_bar + shape_text).properties(
    width=650, height=230, title='Shape of cross-service translation disagreement'
).configure_view(stroke=None)
display(shape_chart)
save_chart(shape_chart, str(FIGURE_DIR / 'disagreement_shape_distribution.png'))

,disagreement_shape,languages,label,pct,annotation
0,fragmented_disagreement,746,Fragmented Disagreement,0.846765,746 (84.7%)
1,dominant_form_with_alternatives,128,Dominant Form With Alternatives,0.145289,128 (14.5%)
2,unanimous,5,Unanimous,0.005675,5 (0.6%)
3,near_unanimous_with_outlier,2,Near Unanimous With Outlier,0.002270,2 (0.2%)


alt.LayerChart(...)

In [3]:
status_shape = patterns.groupby(['comparison_status', 'disagreement_shape']).size().reset_index(name='languages')
status_shape['share'] = status_shape.groupby('comparison_status')['languages'].transform(lambda s: s / s.sum())
status_shape['shape_label'] = status_shape['disagreement_shape'].map(SHAPE_LABELS)

status_shape_chart = alt.Chart(status_shape).mark_bar().encode(
    x=alt.X('comparison_status:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('share:Q', title='Share within comparison status', axis=alt.Axis(format='%')),
    color=alt.Color('shape_label:N', title='Disagreement shape', scale=alt.Scale(scheme='tableau10')),
    order=alt.Order('share:Q', sort='descending'),
).properties(width=620, height=330, title='String distance and cluster shape capture different properties')
display(status_shape_chart)
save_chart(status_shape_chart, str(FIGURE_DIR / 'comparison_status_by_disagreement_shape.png'))

alt.Chart(...)

## 8.3 Who aligns and who becomes an outlier?

I compare exact-form concentration within API LLMs, local LLMs, machine-translation services, and all LLMs. The resulting service-group pattern does not rank those groups. It records whether agreement is concentrated in one part of the infrastructure and identifies singleton service outliers for close reading.

In [4]:
service_counts = patterns['service_group_pattern'].value_counts().rename_axis('service_group_pattern').reset_index(name='languages')
service_counts['label'] = (service_counts['service_group_pattern'].str.replace('_', ' ').str.title()
                           .str.replace('Api', 'API').str.replace('Llm', 'LLM'))
display(service_counts)

service_chart = alt.Chart(service_counts).mark_bar().encode(
    y=alt.Y('label:N', sort='-x', title=None),
    x=alt.X('languages:Q', title='Language-code rows'),
    color=alt.Color('service_group_pattern:N', legend=None, scale=alt.Scale(scheme='set2')),
).properties(width=620, height=220, title='Alignment within API and local LLM groups')
display(service_chart)
save_chart(service_chart, str(FIGURE_DIR / 'service_group_disagreement_patterns.png'))

print('Rows with an MT/LLM dominant-form split:', int(patterns['has_mt_llm_split'].sum()))
print('Rows with at least one singleton service outlier:', int(patterns['singleton_outlier_service_count'].gt(0).sum()))
display(patterns['singleton_outlier_services'].value_counts().head(15).rename_axis('outlier_services').reset_index(name='languages'))

,service_group_pattern,languages,label
0,api_and_local_fragmented,454,API And Local Fragmented
1,mixed_group_alignment,243,Mixed Group Alignment
2,api_aligned_local_fragmented,173,API Aligned Local Fragmented
3,api_and_local_aligned,10,API And Local Aligned
4,local_aligned_api_fragmented,1,Local Aligned API Fragmented


alt.Chart(...)

Rows with an MT/LLM dominant-form split: 11
Rows with at least one singleton service outlier: 129


,outlier_services,languages
0,Gemma; Llama; Mistral; Qwen,27
1,Llama; Mistral; Qwen,22
2,Mistral; Qwen,9
3,Gemma; Llama; Qwen,7
4,Llama; Qwen,6
5,DeepSeek; Llama; Mistral; Qwen,4
6,Gemma; Mistral; Qwen,4
7,EasyNMT; Llama; Mistral; Qwen,3
8,Google Translate; Llama; Mistral; Qwen,2
9,Gemma; Qwen,2


## 8.4 Prompt trajectories

Prompt sensitivity is measured within each LLM service across the four prompt variants. The notebook also records cases where the GitHub prompt introduces source-term echoing that was absent from the minimal prompt, and whether the judge prompt reduces or increases the number of distinct LLM forms. Judge-induced convergence is descriptive: synthesis can harmonize outputs without making them more accurate.

In [5]:
judge_counts = patterns['judge_effect'].value_counts().rename_axis('judge_effect').reset_index(name='languages')
judge_counts['label'] = judge_counts['judge_effect'].str.replace('_', ' ').str.title()

judge_chart = alt.Chart(judge_counts).mark_bar().encode(
    y=alt.Y('label:N', sort='-x', title=None),
    x=alt.X('languages:Q', title='Language-code rows'),
    color=alt.Color('judge_effect:N', legend=None, scale=alt.Scale(scheme='tableau10')),
).properties(width=620, height=150, title='How the judge prompt changes cross-service form diversity')
display(judge_chart)
save_chart(judge_chart, str(FIGURE_DIR / 'judge_prompt_disagreement_effect.png'))

prompt_summary = patterns[[
    'prompt_stable_service_count', 'prompt_sensitive_service_count',
    'github_source_echo_service_count', 'minimal_unique_llm_forms', 'judge_unique_llm_forms',
]].describe().round(2)
display(prompt_summary)
print('Languages where GitHub framing introduces source echo:', int(patterns['github_source_echo_service_count'].gt(0).sum()))

embedding_chart = alt.Chart(patterns.dropna(subset=['mean_embedding_distance'])).mark_boxplot(size=35).encode(
    x=alt.X('disagreement_shape:N', sort=SHAPE_ORDER, title=None, axis=alt.Axis(labelAngle=325)),
    y=alt.Y('mean_embedding_distance:Q', title='Mean pairwise LLM embedding distance'),
    color=alt.Color('disagreement_shape:N', legend=None, scale=alt.Scale(scheme='tableau10')),
).properties(width=620, height=300, title='Embedding dispersion by exact-form disagreement shape')
display(embedding_chart)
save_chart(embedding_chart, str(FIGURE_DIR / 'embedding_dispersion_by_disagreement_shape.png'))

alt.Chart(...)

,prompt_stable_service_count,prompt_sensitive_service_count,github_source_echo_service_count,minimal_unique_llm_forms,judge_unique_llm_forms
count,881.00,881.00,881.00,881.00,881.00
mean,0.87,7.13,0.66,6.78,4.22
std,1.45,1.45,0.92,1.38,1.73
min,0.00,0.00,0.00,1.00,1.00
25%,0.00,7.00,0.00,6.00,3.00
50%,0.00,8.00,0.00,7.00,4.00
75%,1.00,8.00,1.00,8.00,5.00
max,8.00,8.00,5.00,8.00,8.00


Languages where GitHub framing introduces source echo: 373


alt.Chart(...)

[2026-09-24T14:02:59Z WARN  tiny_skia::painter] path stroking failed
[2026-09-24T14:02:59Z WARN  tiny_skia::painter] path stroking failed


## 8.5 External and script evidence

Wikipedia relationships, source retention, and script disagreement remain independent. The heatmap reports how frequently those signals appear within each disagreement shape. It does not treat any signal as proof of correctness, absence, or community acceptance.

In [6]:
evidence = patterns.copy()
evidence['has_source_retention'] = evidence['source_retention_service_count'].gt(0)
evidence['has_singleton_outlier'] = evidence['singleton_outlier_service_count'].gt(0)
evidence['has_prompt_instability'] = evidence['prompt_sensitive_service_count'].gt(0)

signal_map = {
    'has_wikipedia': 'Wikipedia term',
    'has_source_retention': 'Source retention',
    'has_script_disagreement': 'Script disagreement',
    'has_mt_llm_split': 'MT/LLM split',
    'has_singleton_outlier': 'Singleton outlier',
    'has_prompt_instability': 'Prompt instability',
}
evidence_long = (
    evidence.groupby('disagreement_shape')[list(signal_map)].mean()
    .rename(columns=signal_map).reset_index()
    .melt(id_vars='disagreement_shape', var_name='signal', value_name='share')
)
evidence_long['shape_label'] = evidence_long['disagreement_shape'].map(SHAPE_LABELS)

evidence_heat = alt.Chart(evidence_long).mark_rect().encode(
    x=alt.X('signal:N', title=None, axis=alt.Axis(labelAngle=325)),
    y=alt.Y('shape_label:N', sort=[SHAPE_LABELS[s] for s in SHAPE_ORDER], title=None),
    color=alt.Color('share:Q', title='Share', scale=alt.Scale(scheme='blues', domain=[0, 1])),
).properties(width=620, height=230, title='Independent evidence signals by disagreement shape')
evidence_text = evidence_heat.mark_text().encode(
    text=alt.Text('share:Q', format='.0%'),
    color=alt.condition('datum.share > 0.55', alt.value('white'), alt.value('black')),
)
evidence_figure = evidence_heat + evidence_text
display(evidence_figure)
save_chart(evidence_figure, str(FIGURE_DIR / 'evidence_signals_by_disagreement_shape.png'))

alt.LayerChart(...)

## 8.6 What the rationales claim

Rationale signals are aggregated across the English-language prompt variants. They remain claims made by models, not verification. I distinguish definitional registers from translation stances such as asserting an established equivalent, recommending borrowing, proposing coinage, or admitting uncertainty.

In [7]:
rationale_cols = [
    'claims_established_equivalent', 'claims_borrowing',
    'claims_coinage', 'admits_uncertainty',
    'register_compositional_literal', 'register_scholarly_methodological',
    'register_human_centered', 'register_knowledge_generic',
]
rationale_summary = patterns[rationale_cols].mean().sort_values(ascending=False).rename('share_of_rationales').reset_index(name='share_of_rationales').rename(columns={'index': 'signal'})
rationale_summary['label'] = rationale_summary['signal'].str.replace('_', ' ').str.title()
display(rationale_summary)

rationale_chart = alt.Chart(rationale_summary).mark_bar().encode(
    y=alt.Y('label:N', sort='-x', title=None),
    x=alt.X('share_of_rationales:Q', title='Mean share of English rationales', axis=alt.Axis(format='%')),
    color=alt.Color('signal:N', legend=None, scale=alt.Scale(scheme='tableau10')),
).properties(width=620, height=280, title='Definitional and translation stances in model rationales')
display(rationale_chart)
save_chart(rationale_chart, str(FIGURE_DIR / 'rationale_stances_in_disagreement_analysis.png'))

,signal,share_of_rationales,label
0,register_compositional_literal,0.644271,Register Compositional Literal
1,register_knowledge_generic,0.302861,Register Knowledge Generic
2,register_scholarly_methodological,0.253653,Register Scholarly Methodological
3,claims_borrowing,0.236724,Claims Borrowing
4,register_human_centered,0.156625,Register Human Centered
5,claims_established_equivalent,0.125357,Claims Established Equivalent
6,claims_coinage,0.096165,Claims Coinage
7,admits_uncertainty,0.062970,Admits Uncertainty


alt.Chart(...)

### How rationale claims overlap

The prevalence bars do not show whether claims occur in the same rationale. The first heatmap below uses the reviewed rationale-level signal table generated with `disagreement_patterns.csv`. Each cell is the **Jaccard overlap** between two categories: the number of rationales matching both divided by the number matching either. The matrix is symmetric, and the diagonal is 100% by definition. A high value means two regex signals frequently describe the same rationales; it does not mean the claims are equivalent or correct.

The second heatmap connects rationale framing to the external and technical evidence from §8.5. Its cells report the percentage-point difference in a claim's mean language-level prevalence when an evidence signal is present versus absent. Positive values mean the claim is more prevalent among languages carrying that signal; negative values mean it is less prevalent. This is descriptive association across language rows, not evidence that the technical signal caused the rationale claim or that either validates the translation. The present-group count appears in the interactive tooltip; especially imbalanced comparisons require caution. For example, 875 languages show prompt instability, leaving only six in the absent group.

In [8]:
RATIONALE_DETAIL_PATH = os.path.join(EVAL_DIR, 'rationale_signal_detail.csv')
rationale_detail = read_csv_file(RATIONALE_DETAIL_PATH)

rationale_labels = {
    'register_compositional_literal': 'Compositional / literal',
    'register_knowledge_generic': 'Generic knowledge',
    'register_scholarly_methodological': 'Scholarly / methodological',
    'claims_borrowing': 'Borrowing',
    'register_human_centered': 'Human-centered',
    'claims_established_equivalent': 'Established equivalent',
    'claims_coinage': 'Coinage',
    'admits_uncertainty': 'Uncertainty',
}
rationale_order = list(rationale_labels)
rationale_label_order = [rationale_labels[column] for column in rationale_order]

for column in rationale_order:
    rationale_detail[column] = (
        rationale_detail[column].astype(str).str.strip().str.lower().eq('true')
    )

overlap_rows = []
for row_signal in rationale_order:
    for column_signal in rationale_order:
        intersection = int((rationale_detail[row_signal] & rationale_detail[column_signal]).sum())
        union = int((rationale_detail[row_signal] | rationale_detail[column_signal]).sum())
        overlap_rows.append({
            'row_claim': rationale_labels[row_signal],
            'column_claim': rationale_labels[column_signal],
            'intersection': intersection,
            'union': union,
            'jaccard': intersection / union if union else np.nan,
        })
overlap_df = pd.DataFrame(overlap_rows)

overlap_base = alt.Chart(overlap_df).encode(
    x=alt.X('column_claim:N', sort=rationale_label_order, title=None, axis=alt.Axis(labelAngle=-42)),
    y=alt.Y('row_claim:N', sort=rationale_label_order, title=None),
    tooltip=['row_claim:N', 'column_claim:N', 'intersection:Q', 'union:Q',
             alt.Tooltip('jaccard:Q', format='.1%', title='Jaccard overlap')],
)
overlap_heat = overlap_base.mark_rect().encode(
    color=alt.Color('jaccard:Q', title='Jaccard overlap',
                    scale=alt.Scale(scheme='blues', domain=[0, 1])),
)
overlap_text = overlap_base.mark_text(fontSize=10).encode(
    text=alt.Text('jaccard:Q', format='.0%'),
    color=alt.condition(alt.datum.jaccard > 0.55, alt.value('white'), alt.value('black')),
)
rationale_overlap_figure = (overlap_heat + overlap_text).properties(
    width=520, height=460,
    title=alt.TitleParams(
        'Rationale-Level Overlap Among Definitional and Translation Claims',
        subtitle='Jaccard overlap: rationales matching both signals / rationales matching either signal.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
display(rationale_overlap_figure)
save_chart(rationale_overlap_figure, str(FIGURE_DIR / 'rationale_claim_overlap_heatmap.png'))

def as_bool_signal(series):
    if str(series.dtype) == 'bool':
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes'})

intersection_rows = []
for evidence_column, evidence_label in signal_map.items():
    present = as_bool_signal(evidence[evidence_column])
    for rationale_column in rationale_order:
        present_mean = patterns.loc[present, rationale_column].mean()
        absent_mean = patterns.loc[~present, rationale_column].mean()
        intersection_rows.append({
            'evidence_signal': evidence_label,
            'rationale_claim': rationale_labels[rationale_column],
            'present_mean': present_mean,
            'absent_mean': absent_mean,
            'difference': present_mean - absent_mean,
            'n_signal_present': int(present.sum()),
        })
claim_evidence_df = pd.DataFrame(intersection_rows)
difference_limit = float(np.ceil(claim_evidence_df['difference'].abs().max() * 20) / 20)

intersection_base = alt.Chart(claim_evidence_df).encode(
    x=alt.X('evidence_signal:N', sort=list(signal_map.values()), title=None,
            axis=alt.Axis(labelAngle=-38)),
    y=alt.Y('rationale_claim:N', sort=rationale_label_order, title=None),
    tooltip=['rationale_claim:N', 'evidence_signal:N', 'n_signal_present:Q',
             alt.Tooltip('present_mean:Q', format='.1%', title='Mean with signal'),
             alt.Tooltip('absent_mean:Q', format='.1%', title='Mean without signal'),
             alt.Tooltip('difference:Q', format='+.1%', title='Difference')],
)
intersection_heat = intersection_base.mark_rect().encode(
    color=alt.Color('difference:Q', title='Difference',
                    scale=alt.Scale(scheme='redblue', reverse=True,
                                    domain=[-difference_limit, difference_limit], domainMid=0)),
)
intersection_text = intersection_base.mark_text(fontSize=10).encode(
    text=alt.Text('difference:Q', format='+.0%'),
    color=alt.condition(f'abs(datum.difference) > {difference_limit * 0.55}',
                        alt.value('white'), alt.value('black')),
)
claim_evidence_figure = (intersection_heat + intersection_text).properties(
    width=620, height=360,
    title=alt.TitleParams(
        'Rationale Claims × External and Technical Evidence',
        subtitle='Percentage-point difference in mean claim prevalence: signal present minus absent.',
        subtitleColor='#666', subtitleFontSize=11,
    ),
)
display(claim_evidence_figure)
save_chart(claim_evidence_figure, str(FIGURE_DIR / 'rationale_claims_by_evidence_signals_heatmap.png'))

alt.LayerChart(...)

alt.LayerChart(...)

## 8.7 Cases for close reading

The final score is only a sorting device. It prioritizes rows where several different forms of evidence intersect: fragmentation, service outliers, prompt sensitivity, script disagreement, source retention, MT/LLM division, Wikipedia evidence, and explicit uncertainty or coinage claims. These are strong candidates for qualitative review because no single signal explains them.

In [9]:
review = patterns.copy()
review['review_interest_score'] = (
    review['disagreement_shape'].eq('fragmented_disagreement').astype(int)
    + review['singleton_outlier_service_count'].gt(0).astype(int)
    + review['prompt_sensitive_service_count'].ge(6).astype(int)
    + review['has_script_disagreement'].astype(int)
    + review['source_retention_service_count'].gt(0).astype(int)
    + review['has_mt_llm_split'].astype(int)
    + review['has_wikipedia'].astype(int)
    + review['admits_uncertainty'].gt(0).astype(int)
    + review['claims_coinage'].gt(0).astype(int)
)

review_cols = [
    'language_code', 'language_name', 'language_family', 'comparison_status',
    'disagreement_shape', 'dominant_form_share', 'service_group_pattern',
    'singleton_outlier_services', 'prompt_sensitive_service_count',
    'judge_effect', 'wikipedia_relationship', 'source_retention_service_count',
    'has_script_disagreement', 'has_mt_llm_split', 'claims_borrowing',
    'claims_coinage', 'admits_uncertainty', 'review_interest_score',
]
close_reading_candidates = review.sort_values(
    ['review_interest_score', 'n_unique_raw', 'prompt_sensitive_service_count'],
    ascending=False,
)[review_cols]
display(close_reading_candidates.head(30))

,language_code,language_name,language_family,comparison_status,disagreement_shape,dominant_form_share,service_group_pattern,singleton_outlier_services,prompt_sensitive_service_count,judge_effect,wikipedia_relationship,source_retention_service_count,has_script_disagreement,has_mt_llm_split,claims_borrowing,claims_coinage,admits_uncertainty,review_interest_score
457,lus,Mizo,Sino-Tibetan languages,substantive_variation,fragmented_disagreement,0.400000,api_aligned_local_fragmented,NaN,7,judge_induced_convergence,no_wikipedia_term,6,True,True,0.500000,0.125000,0.083333,7
759,ti,Tigrinya,Afro-Asiatic languages,substantive_variation,fragmented_disagreement,0.100000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.166667,0.125000,0.125000,6
664,sat,Santali,Austro-Asiatic languages,substantive_variation,fragmented_disagreement,0.111111,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.208333,0.041667,0.041667,6
137,chp,Chipewyan,Athabaskan-Eyak-Tlingit,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.083333,0.083333,0.083333,6
184,dtm,Tomo Kan Dogon,Dogon,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.291667,0.041667,0.166667,6
216,fan,Fang,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.250000,0.083333,0.083333,6
328,iu,Inuktitut,Eskimo-Aleut languages,substantive_variation,fragmented_disagreement,0.222222,mixed_group_alignment,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.125000,0.041667,0.041667,6
351,kcg,Tyap,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.208333,0.041667,0.125000,6
438,lki,Laki,Indo-European languages,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,2,True,False,0.333333,0.125000,0.041667,6
449,lsm,Saamia,Atlantic-Congo,substantive_variation,fragmented_disagreement,0.125000,api_and_local_fragmented,NaN,8,judge_induced_convergence,no_wikipedia_term,1,True,False,0.304348,0.086957,0.043478,6


## 8.8 Interpretation

The central result is not that disagreement belongs to one of several stable types. It is that different dimensions overlap unevenly. A language can show substantive string variation while still having a dominant form; a Wikipedia term can anchor either the dominant or an alternative cluster; a judge prompt can reduce diversity while rationales continue to express uncertainty; and one service can be a script outlier within an otherwise stable cluster.

This framework therefore separates three analytical moves:

1. **Measure degree** with exact, near-form, and substantive comparison.
2. **Describe structure** with cluster shape, service grouping, and prompt trajectory.
3. **Interpret selected cases** using external evidence, rationale claims, and language-specific research.

The first two can be automated transparently. The third remains a scholarly judgment.